LN Project - Baseline

Imports

In [1]:
# Data manipulation and regex
import pandas as pd
import re
import csv

# Text processing
from nltk.stem import PorterStemmer

# Machine Learning: Feature extraction, Model, and Evaluation
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

Data Loading & Cleaning

In [2]:
# Load the dataset
df = pd.read_csv('train.csv', sep=';', on_bad_lines='skip', quotechar='"')

# Clean column names and target labels
df.columns = df.columns.str.strip()
df['medical_specialty'] = df['medical_specialty'].str.strip()

# Keep only the 12 valid classes (filtering out bad parsing artifacts)
valid_classes = df['medical_specialty'].value_counts()
valid_classes = valid_classes[valid_classes > 10].index.tolist()
df = df[df['medical_specialty'].isin(valid_classes)].copy()

# Drop rows missing the target or the description
df = df.dropna(subset=['description', 'medical_specialty'])

print(f"Dataset shape after cleaning: {df.shape}")

Dataset shape after cleaning: (2613, 5)


Text Pre-processing

In [3]:
# Initialize the stemmer
stemmer = PorterStemmer()

def preprocess_baseline(text):
    if not isinstance(text, str):
        return ""
    # 1. Lowercase
    text = text.lower()
    # 2. Remove punctuation
    text = re.sub(r'[^\w\s]', '', text)
    # 3. Apply Stemming
    return " ".join([stemmer.stem(word) for word in text.split()])

# Apply the pre-processing to the description column
df['processed_desc'] = df['description'].apply(preprocess_baseline)

print("Sample processed description:")
print(df['processed_desc'].iloc[0])

Sample processed description:
chest pain possibl syncop spell she ha been have multipl cardiovascular complaint includ chest pain which feel like cramp and sometim like a dull ach which will last all day long


Train / Validation Split

In [4]:
X_train, X_val, y_train, y_val = train_test_split(
    df['processed_desc'], 
    df['medical_specialty'], 
    test_size=0.2, 
    random_state=42, 
    stratify=df['medical_specialty']
)

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")

Training samples: 2090
Validation samples: 523


TF-IDF Feature Extraction

In [5]:
# Initialize and fit the TF-IDF Vectorizer
vectorizer = TfidfVectorizer()

# Fit on training data and transform both train and val sets
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)

print(f"TF-IDF Vocabulary Size: {len(vectorizer.vocabulary_)}")

TF-IDF Vocabulary Size: 5945


Model Training

In [6]:
# Initialize and train the baseline model
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

print("Multinomial Naive Bayes model trained successfully!")

Multinomial Naive Bayes model trained successfully!


Evaluation

In [7]:
# Make predictions
y_pred = nb_model.predict(X_val_tfidf)

# Calculate Accuracy
acc = accuracy_score(y_val, y_pred)

print(f"--- BASELINE RESULTS ---")
print(f"Overall Accuracy: {acc * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(y_val, y_pred))

--- BASELINE RESULTS ---
Overall Accuracy: 36.52%

Classification Report:
                          precision    recall  f1-score   support

Cardiovascular-Pulmonary       0.48      0.17      0.25        60
             Dermatology       0.00      0.00      0.00         5
        Gastroenterology       0.00      0.00      0.00        38
        General Medicine       1.00      0.17      0.29        41
               Neurology       0.00      0.00      0.00        36
            Neurosurgery       0.00      0.00      0.00        16
   Obstetrics-Gynecology       0.00      0.00      0.00        27
           Ophthalmology       0.00      0.00      0.00        14
              Orthopedic       0.23      0.05      0.08        59
   Psychiatry-Psychology       0.00      0.00      0.00         8
               Radiology       0.00      0.00      0.00        44
                 Surgery       0.37      0.98      0.54       175

                accuracy                           0.37       523
